#### Group Fairness 모델
- FairGNN (Adversarial Learning): 대표
    - FairVGNN (Adversarial Learning)

- EDITS (Edge Rewiring): 대표
    - UGE (Edge Rewiring) : 제외
    - FairEdit (Edge Rewiring)
    - GEAR (Edge Rewiring)

- FairWalk (Rebalancing): 대표
    - CrossWalk (Rebalancing)

- NIFTY (Optimization with Regularization)

- FairGB

In [ ]:
import pandas as pd

### Avg.

In [ ]:
df1 = pd.read_csv('./outputs/exp_fairgate_gcn.csv')
df1 = df1[['model', 'dataset', 'acc_mean', 'roc_auc_mean', 'dp_mean', 'eo_mean']]

df2 = pd.read_csv('./outputs/compare/exp_baselines.csv')
df2 = df2[['model', 'dataset', 'acc_mean', 'roc_auc_mean', 'dp_mean', 'eo_mean']]

df = pd.concat([df1, df2])
df_rank_v1 = df[df["model"] != "GCN"].copy()  # baseline 제외

df_rank = df_rank_v1[df_rank_v1['dataset'] != 'recidivism'].copy() # recidivism 제외시
# df_rank = df_rank_v1.copy() # 제외 아닐 시

# dataset별 rank 계산
df_rank["rank_Acc"] = df_rank.groupby("dataset")["acc_mean"].rank(ascending=False, method="average")
df_rank["rank_AUC"] = df_rank.groupby("dataset")["roc_auc_mean"].rank(ascending=False, method="average")
df_rank["rank_DP"]  = df_rank.groupby("dataset")["dp_mean"].rank(ascending=True,  method="average")
df_rank["rank_EO"]  = df_rank.groupby("dataset")["eo_mean"].rank(ascending=True,  method="average")

# accuracy / fairness 평균 rank
df_rank["rank_acc_avg"] = df_rank[["rank_Acc", "rank_AUC"]].mean(axis=1)
df_rank["rank_fair_avg"] = df_rank[["rank_DP", "rank_EO"]].mean(axis=1)

# 전체 dataset에 대한 모델별 평균 rank
model_order = ['FairGNN', 'FairVGNN', 
               'FairEdit', 'EDITS', 
               'FairWalk', 'CrossWalk', 
               'FairGB', 'NIFTY', 
               'FairGT', 'FairGate']

summary = (
    df_rank.groupby("model")[["rank_Acc", "rank_AUC", "rank_DP", "rank_EO",
                              "rank_acc_avg", "rank_fair_avg"]]
    .mean()
)

summary = summary.reindex(model_order)

print(summary.round(4))

In [ ]:
# df1 = pd.read_csv('./outputs/ours/exp_fairgate_fiw_v1.csv')
df1 = pd.read_csv('./outputs/exp_fairgate_new.csv')
df1 = df1[['model', 'dataset', 'acc_mean', 'roc_auc_mean', 'dp_mean', 'eo_mean']]

df2 = pd.read_csv('./outputs/compare/exp_baselines.csv')
df2 = df2[['model', 'dataset', 'acc_mean', 'roc_auc_mean', 'dp_mean', 'eo_mean']]

df = pd.concat([df1, df2], ignore_index=True)


df_rank_v1 = df[df["model"] != "GCN"].copy()   # baseline 제외
# df_rank = df_rank_v1[df_rank_v1['dataset'] != 'recidivism'].copy() # recidivism 제외
df_rank = df_rank_v1.copy()


df_rank["rank_Acc"] = df_rank.groupby("dataset")["acc_mean"].rank(
    ascending=False, method="average"
)
df_rank["rank_AUC"] = df_rank.groupby("dataset")["roc_auc_mean"].rank(
    ascending=False, method="average"
)
df_rank["rank_DP"] = df_rank.groupby("dataset")["dp_mean"].rank(
    ascending=True, method="average"
)
df_rank["rank_EO"] = df_rank.groupby("dataset")["eo_mean"].rank(
    ascending=True, method="average"
)

# metric-group average rank
df_rank["rank_acc_avg"] = df_rank[["rank_Acc", "rank_AUC"]].mean(axis=1)
df_rank["rank_fair_avg"] = df_rank[["rank_DP", "rank_EO"]].mean(axis=1)


df_tmp = (
    df_rank.groupby(["model", "dataset"], as_index=False)[
        ["rank_Acc", "rank_AUC", "rank_DP", "rank_EO",
         "rank_acc_avg", "rank_fair_avg"]
    ]
    .mean()
)

summary = (
    df_tmp.groupby("model")[
        ["rank_Acc", "rank_AUC", "rank_DP", "rank_EO",
         "rank_acc_avg", "rank_fair_avg"]
    ]
    .mean()
)

# dataset coverage count
summary["num_datasets"] = df_tmp.groupby("model")["dataset"].nunique()

model_order = [
    'FairGNN', 'FairVGNN',
    'FairEdit', 'EDITS',
    'FairWalk', 'CrossWalk',
    'FairGB', 'NIFTY',
    'FairGT', 'FairGate'
]

summary = summary.reindex(model_order)

# =========================
# 6. Print
# =========================
print(summary.round(2))

### ours

In [ ]:
file_name ='exp_fairgate_gcn'

datasets = [
    'pokec_z', 'pokec_n', 
    'pokec_z_g', 'pokec_n_g',
    'credit', 'recidivism', 'income', 
    'german', 
    'nba'
             ]

df = pd.read_csv(f'./outputs/{file_name}.csv')
df = df[[
    'model', 'dataset',
    'acc_mean', 
    'acc_std', 
    'roc_auc_mean', 
    'roc_auc_std',
    'dp_mean', 
    'dp_std', 
    'eo_mean', 
    'eo_std', 
    # 'time_sec_mean', 'time_sec_std'
]]

df = df.copy()
df['dataset'] = pd.Categorical(
    df['dataset'],
    categories=datasets,
    ordered=True
)

df.sort_values('dataset')
# [df.dataset == 'german']

In [ ]:
file_name ='exp_fairgate_fiw_v1'

datasets = ['pokec_z', 'pokec_n', 'pokec_z_g', 'pokec_n_g',
             'credit', 'recidivism', 'income', 'german', 'nba']

df = pd.read_csv(f'./outputs/ours/{file_name}.csv')
df = df[[
    'model', 'dataset',
    'acc_mean', 
    # 'acc_std', 
    'roc_auc_mean', 
    # 'roc_auc_std',
    'dp_mean', 
    # 'dp_std', 
    'eo_mean', 
    # 'eo_std', 
    # 'time_sec_mean', 'time_sec_std'
]]

df = df.copy()
df['dataset'] = pd.Categorical(
    df['dataset'],
    categories=datasets,
    ordered=True
)

df.sort_values('dataset')
# [df.dataset == 'nba']

In [ ]:
df = pd.read_csv(f'./outputs/tune_lambda.csv')
df = df[[
    'model', 'dataset',
    'acc_mean', 
    # 'acc_std', 
    'roc_auc_mean', 
    # 'roc_auc_std',
    'dp_mean', 
    # 'dp_std', 
    'eo_mean', 
    # 'eo_std', 
    # 'time_sec_mean', 'time_sec_std'
]]
df


### baselines

In [ ]:
file_name ='exp_baselines'
df_v2 = pd.read_csv(f'./outputs/compare/{file_name}.csv')
df_v2 = df_v2[[
    'model', 'dataset',
    'acc_mean', 
    'acc_std', 
    'roc_auc_mean',
    'roc_auc_std',
    'dp_mean',
    'dp_std',
    'eo_mean', 
    'eo_std', 
    # 'time_sec_mean', 
    # 'time_sec_std'
]]

datasets = ['pokec_z', 'pokec_n', 'pokec_z_g', 'pokec_n_g',
             'credit', 'recidivism', 'income', 'german', 'nba']
model_order = ['FairGNN', 'FairVGNN', 'FairEdit', 'EDITS',
               'FairWalk', 'CrossWalk', 'FairGB', 'NIFTY', 'FairGT']

filtered_df = df_v2.copy()
filtered_df['model'] = pd.Categorical(
    filtered_df['model'],
    categories=model_order,
    ordered=True
)
filtered_df['dataset'] = pd.Categorical(
    filtered_df['dataset'],
    categories=datasets,
    ordered=True
)

filtered_df = filtered_df.sort_values(['model', 'dataset'])
# filtered_df[filtered_df.dataset == 'recidivism']
filtered_df

In [ ]:
file_name ='exp_baselines_sc'
df_v2 = pd.read_csv(f'./outputs/{file_name}.csv')
df_v2 = df_v2[[
    'model', 'dataset',
    'acc_mean', 
    # 'acc_std', 
    'roc_auc_mean',
    # 'roc_auc_std',
    'dp_mean',
    # 'dp_std',
    'eo_mean', 
    # 'eo_std', 
    # 'time_sec_mean', 
    # 'time_sec_std'
]]

datasets = ['pokec_z', 'pokec_n', 'pokec_z_g', 'pokec_n_g',
             'credit', 'recidivism', 'income', 'german', 'nba']
model_order = ['FairGNN', 'FairVGNN', 'FairEdit', 'EDITS',
               'FairWalk', 'CrossWalk', 'FairGB', 'NIFTY', 'FairGT']

filtered_df = df_v2.copy()
filtered_df['model'] = pd.Categorical(
    filtered_df['model'],
    categories=model_order,
    ordered=True
)
filtered_df['dataset'] = pd.Categorical(
    filtered_df['dataset'],
    categories=datasets,
    ordered=True
)

filtered_df = filtered_df.sort_values(['model', 'dataset'])
filtered_df[filtered_df.model == 'CrossWalk']
# filtered_df

### gcn

In [ ]:
file_name ='exp_gcn'
df_ab = pd.read_csv(f'./outputs/compare/{file_name}.csv')
# df_ab = df_ab[df_ab.dataset == 'recidivism']
df_ab = df_ab.drop(['stage_label', 'run', 'seed'], axis=1)

df_ab.groupby(['dataset', 'stage']).agg(['mean', 'std']).round(4)